In [1]:
# libraries
import asyncio
import os
from datetime import datetime
from dataclasses import dataclass
import numpy as np
from nea_tools import connect, disconnect
import matplotlib.pyplot as plt
import h5py

In [2]:
# parameters scan mid-IR
DISTANCE_X = 5000 # scan range in fast axis (x), in nm
DISTANCE_Y = 5000 # scan range in slow axis (y), in nm
DISTANCE_Z = 2500 # scan range in slow axis (z), in nm
RESOLUTION_X = 10 # pixels in fast axis (x)
RESOLUTION_Y = 10 # pixels in slow axis (y)
RESOLUTION_Z = 3 # pixels in slow axis (z)
SLEEP_TIMER = 0.2 # time to wait after each movement, in s

In [ ]:
# parameters scan VIS
DISTANCE_X = 8000 # scan range in fast axis (x), in nm
DISTANCE_Y = 8000 # scan range in slow axis (y), in nm
DISTANCE_Z = 0 # scan range in slow axis (z), in nm
RESOLUTION_X = 20 # pixels in fast axis (x)
RESOLUTION_Y = 20 # pixels in slow axis (y)
RESOLUTION_Z = 1 # pixels in slow axis (z)
SLEEP_TIMER = 0.2 # time to wait after each movement, in s

In [ ]:
# definitions
@dataclass
class ScanResult:
    """Dataclass to combine measured data of mirror scan"""
    o0:np.ndarray
    o1:np.ndarray
    o2:np.ndarray
    o3:np.ndarray
    o4:np.ndarray
    o5:np.ndarray
    coordinates:np.ndarray
    """Measured mirror xyz coordinates in nm"""

    def __iter__(self):
        return iter((self.o0,self.o1,self.o2,self.o3,self.o4,self.o5))

async def scan_mirror(dx, dy, dz, res_x, res_y, res_z, sleep_timer = 0.3):
    """
    Perform a line-by-line scanning movement of the parabolic mirror
    to spatially map the laser intensity in XY plane. Scanning area is a rectangular
    with current mirror motor position as center. Returns measured 
    optical amplitudes as numpy arrays in a dict. Dict keys are indeces of correspondic
    optical harmonic

    Args:
        dx (float): scan range in x axis, in nm
        dy (float): scan range in y axis, in nm
        dz (float): scan range in y axis, in nm
        res_x (int): number of steps in x axis
        res_y (int): number of steps in y axis
        res_z (int): number of steps in z axis
        sleep_timer (float): time to wait after each mirror movement

    Returns:
        results (ScanResult): containss measured optical amplitudes and corresponding xyz
                mirror coordinates. 
    """
    amp:"dict[int,np.ndarray]" = {}
    phase:"dict[int,np.ndarray]" = {}
    for harmonic in range(6):
        amp[harmonic] = np.zeros((res_z,res_y,res_x))
        phase[harmonic] = np.zeros((res_z,res_y,res_x))
    
    # correct for offset in start position
    Xoff=0; Yoff=1000; Zoff=0 #nm
    
    coords = np.zeros((res_z,res_y,res_x,3))
    with Mirror() as mirror:
        mirror.go_relative(-dx/2-Xoff,-dy/2-Yoff,-dz/2-Zoff)
        await mirror.await_async()
        await asyncio.sleep(sleep_timer)
        x0,y0,z0 = mirror.absolute_position

        for iz in range(res_z):
            for iy in range(res_y):
                for ix in range(res_x):
                    x1,y1,z1 = coords[iz,iy,ix] = mirror.absolute_position
                    print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm, z={int(z1-z0)} nm, O2A={round(context.Microscope.Py.OpticalAmplitude[2],3)} mV")
                    for harmonic in range(6):
                        amp[harmonic][iz,iy,ix] = context.Microscope.Py.OpticalAmplitude[harmonic]
                        phase[harmonic][iz,iy,ix] = context.Microscope.Py.OpticalAmplitude[harmonic]
                    
                    #mirror.go_relative(dx/res_x*(res_x+1)/res_x,0,0) # stepwise movement in x
                    mirror.go_relative(dx/res_x,0,0) # stepwise movement in x
                    await mirror.await_async()
                    await asyncio.sleep(sleep_timer)
                    
                    
                # measure how far mirror actually moved
                # move back in x this much, could be more or less than dx
                mirror.go_relative(x0-x1,dy/res_y,0)
                #mirror.go_relative(x0-x1,dy/res_y*(res_y+1)/res_y,0) # stepwise movement in y and back to x0
                await mirror.await_async()
                await asyncio.sleep(sleep_timer)
                x1,y1,z1 = mirror.absolute_position
                print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm, z={int(z1-z0)} nm")

            # measure how far mirror actually moved
            # move back in y this much, could be more or less than dx
            mirror.go_relative(x0-x1,y0-y1,dz/res_z)
            #mirror.go_relative(x0-x1,y0-y1,dz/res_z*(res_z+1)/res_z) # stepwise movement in z and back to x0,y0
            await mirror.await_async()
            await asyncio.sleep(sleep_timer)
            x1,y1,z1 = mirror.absolute_position
            print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm, z={int(z1-z0)} nm")

        # move back to approx center
        mirror.go_relative(-dx/2,-dy/2,-dz/2)
        await mirror.await_async()
        await asyncio.sleep(sleep_timer)

    amp_result = ScanResult(amp[0],amp[1],amp[2],amp[3],amp[4],amp[5],coords)
    phase_result = ScanResult(phase[0],phase[1],phase[2],phase[3],phase[4],phase[5],coords)
    return amp_result,phase_result

In [3]:
from neaspec_functions import open_nea, close_nea
open_nea()

ModuleNotFoundError: No module named 'Nea'

In [4]:
#open microscope
import asyncio
import nest_asyncio
import nea_tools

path_to_dll = ""
fingerprint = None
host = 'nea-server'
    
# nea_tools.set_output(None) # turn off logging
    
# connecting and creating module neaspec on success
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)
loop.run_until_complete(nea_tools.connect(host,fingerprint,
                                        path_to_dll))

from neaspec import context
from nea_tools.microscope.motors import Mirror

Trying to download from nea-server
12:25:16.272 Nea.Client.Database Connecting to: Username=neaspec;Database=neaspec;Host=nea-server;Port=5432 
12:25:16.825 Nea.Client.Settings Configuration in local file C:\Users\neaspec\AppData\Local\DefaultDomain_Path_fxmicv33gyl0u4t52k12yjhbwjqhjroe\1.0.0.0\user.config 
12:25:16.851 Nea.Client.Settings Provided by database group: Nea.Client.Hardware.Microscope.Properties.Settings 
12:25:17.154 Nea.Client.Database Cache provides settings of group Nea.Client.Hardware.Camera.Properties.Settings 
12:25:17.154 Nea.Client.Settings Provided by database group: Nea.Client.Hardware.Camera.Properties.Settings 
12:25:17.311 Nea.Client.SDK Waiting for initialization... 
12:25:17.483 Nea.Client.Hardware.Microscope Failed to resolve 10.227.127.150 - No such host is known. System.Net.Sockets.SocketException (0x00002AF9): No such host is known.
   at System.Net.Dns.GetHostEntryOrAddressesCore(IPAddress address, Boolean justAddresses, AddressFamily addressFamily, Nu

In [ ]:
from galvo_functions import open_galvo, Galvo
# init galvo
ret=open_galvo()
MyLS=Galvo('galvomotor\cal_files')

In [ ]:
# move laser scanner to specific position
X=0; Y=0 # in focal plane [nm]
MyLS.Move(X,Y)

In [ ]:
# scan
NOW = datetime.now().strftime("%y%m%d")
#NOW = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
DIR='C:\\Users\\neaspec\\Documents\\Philippe\\'+NOW+'-test-parabola-scan'
#sample_name='pscan-sfg_nftir-tip_785nm-100muW_qcl_5x2s_01'
#FNAME='SiRef_parabola_SNOM-MIR_5000x5000x2500_10x10x3_02s_vis-optz_bis'
#FNAME='SiRef_parabola_SNOM-VIS_4000x4000x1000_10x10x3_02s_galvo-x_2000-y_0'
FNAME=f'SiRef_785nm-8mW_parabola_SNOM_{DISTANCE_X}x{DISTANCE_Y}x{DISTANCE_Z}_{RESOLUTION_X}x{RESOLUTION_Y}x{RESOLUTION_Z}'


#DIR= os.path.join(os.environ["HOMEPATH"],"Desktop","ParabolaScan") # directory to save images 
#CMAP = "jet" # color scheme for generated images, see https://matplotlib.org/stable/users/explain/colors/colormaps.html
#FNAME = "ParabolaScan" # beginning of file name of each image

try:
#print('hello')
    if not os.path.exists(DIR):
        os.mkdir(DIR)
    if not os.path.isdir(DIR):
        raise IOError(f"{DIR} is not a directory")
    amp,phase = asyncio.run(scan_mirror(DISTANCE_X,DISTANCE_Y,DISTANCE_Z,RESOLUTION_X,RESOLUTION_Y, RESOLUTION_Z, SLEEP_TIMER))
    
    with h5py.File(os.path.join(DIR,f"{FNAME}_{NOW}.h5"),'w') as file:
        for harmonic, (a,p) in enumerate(zip(amp,phase)):
            #file.create_dataset(f"O{harmonic}",data=a*np.exp(1j*p))
            file.create_dataset(f"O{harmonic}",data=a)
        file.create_dataset("coordinates",data=amp.coordinates)

    #for harmonic, array in enumerate(amp):
        #plt.imsave(os.path.join(DIR,f"{FNAME}_O{harmonic}A_{NOW}.png"),array ,cmap=CMAP)

finally: 
    print('Parabola scan done')

In [ ]:
for harmonic, array in enumerate(amp):
    plt.imsave(os.path.join(DIR,f"{FNAME}_O{harmonic}A_{NOW}.png"),array ,cmap=CMAP)

In [ ]:
# Don't forget to disconnect
print('\nDisconnecting')
nea_tools.disconnect()

In [ ]:
close_nea()